# 03 Baseline Modelling – Configurable Pipeline

## Purpose

This notebook trains and evaluates the same five baseline model families across the three research datasets:

- German Credit
- Pima Diabetes
- COMPAS

The notebook is designed as a reusable pipeline.  
Only the `ACTIVE_DATASET` value in the configuration block should be changed.

The goal is to ensure that from baseline modelling onwards, all datasets are evaluated using the same experimental framework, same model families, same core hyperparameters, same evaluation metrics, and same output structure.

## Key Design Principle

`01` EDA and `02` preprocessing remain dataset-specific.

From `03` onwards, the pipeline is standardised.

This means:

- the same model families are trained;
- the same model settings are used;
- the same train/test evaluation metrics are calculated;
- the same 10-fold cross-validation design is used;
- the same model artefacts are saved;
- the same downstream file naming structure is maintained.

Dataset-specific differences are controlled only through the configuration block.

In [1]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# Select only one active dataset.
ACTIVE_DATASET = "german_credit"
# ACTIVE_DATASET = "pima_diabetes"
# ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "target_name": "target",
        "positive_class_label": "Default",
        "negative_class_label": "Non-default",
        "results_folder": "german_credit",
        "preprocessing_folder": "02_data_preprocessing_german_credit",
        "baseline_folder": "03_baseline_modeling_german_credit",
        "business_score_weights": {
            "recall": 0.50,
            "roc_auc": 0.30,
            "precision": 0.20
        },
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age", "Sex"],
        "business_score_reason": (
            "German Credit prioritises recall because failing to identify likely default cases "
            "is costly, while ROC-AUC and precision remain important for ranking and false-positive control."
        )
    },

    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "target_name": "target",
        "positive_class_label": "Diabetes",
        "negative_class_label": "No diabetes",
        "results_folder": "pima_diabetes",
        "preprocessing_folder": "02_data_preprocessing_pima_diabetes",
        "baseline_folder": "03_baseline_modeling_pima_diabetes",
        "business_score_weights": {
            "recall": 0.50,
            "roc_auc": 0.30,
            "precision": 0.20
        },
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age"],
        "business_score_reason": (
            "Pima Diabetes prioritises recall because missing likely diabetes cases is clinically risky, "
            "while ROC-AUC and precision remain important for ranking and false-positive control."
        )
    },

    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "target_name": "target",
        "positive_class_label": "Recidivism",
        "negative_class_label": "No recidivism",
        "results_folder": "compas",
        "preprocessing_folder": "02_data_preprocessing_compas",
        "baseline_folder": "03_baseline_modeling_compas",
        "business_score_weights": {
            "recall": 0.30,
            "roc_auc": 0.40,
            "precision": 0.30
        },
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["age", "sex", "race"],
        "business_score_reason": (
            "COMPAS balances recall for public safety, precision for discrimination-control risk, "
            "and ROC-AUC for threshold-independent ranking ability."
        )
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]

print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])
print("Positive class:", CONFIG["positive_class_label"])
print("Business score weights:", CONFIG["business_score_weights"])

Active dataset: German Credit
Dataset name: german_credit
Positive class: Default
Business score weights: {'recall': 0.5, 'roc_auc': 0.3, 'precision': 0.2}


In [17]:
# =========================================================
# 1. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from sklearn.base import clone
from pathlib import Path
from datetime import datetime
import json
import time
import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("XGBoost available:", XGBOOST_AVAILABLE)

XGBoost available: True


In [3]:
# =========================================================
# 2. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["baseline_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

PREPROCESSING_RESULTS_PATH = RESULTS_ROOT / CONFIG["preprocessing_folder"]
PREPROCESSING_DATASETS_PATH = PREPROCESSING_RESULTS_PATH / "datasets"
PREPROCESSING_MODELS_PATH = PREPROCESSING_RESULTS_PATH / "models"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
MODELS_PATH = RESULTS_PATH / "models"
FIGURES_PATH = RESULTS_PATH / "figures"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [
    RESULTS_PATH,
    TABLES_PATH,
    DATASETS_PATH,
    MODELS_PATH,
    FIGURES_PATH,
    LOGS_PATH
]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("Preprocessing datasets:", PREPROCESSING_DATASETS_PATH)
print("Baseline output:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/german_credit
Preprocessing datasets: /content/drive/MyDrive/Master_Research/07_Results/german_credit/02_data_preprocessing_german_credit/datasets
Baseline output: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit


In [4]:
# =========================================================
# 3. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)

    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()

    return pd.read_csv(path)


def load_target_series(path):
    target_df = read_csv_safely(path, "target file")

    if "target" in target_df.columns:
        return target_df["target"].astype(int)

    if CONFIG["target_name"] in target_df.columns:
        return target_df[CONFIG["target_name"]].astype(int)

    if target_df.shape[1] == 1:
        return target_df.iloc[:, 0].astype(int)

    raise ValueError(
        f"Could not identify target column in {path}. "
        f"Available columns: {target_df.columns.tolist()}"
    )


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH

    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)

    output_path = folder / filename
    df.to_csv(output_path, index=index)

    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH

    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)

    output_path = folder / filename

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    print(f"Saved: {output_path}")
    return output_path


def get_model_probabilities(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))

    preds = model.predict(X)
    return preds.astype(float)


def calculate_binary_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "average_precision": average_precision_score(y_true, y_proba),
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
    }


def business_priority_score(row, weights):
    return (
        weights["recall"] * row["recall"] +
        weights["roc_auc"] * row["roc_auc"] +
        weights["precision"] * row["precision"]
    )

## 4. Load Preprocessed Modelling Inputs

The notebook expects `02_data_preprocessing` to have saved both original and preprocessed train/test splits.

- Preprocessed data is used for estimator-only models.
- Original data plus fitted preprocessor is used to create full pipeline models.

In [34]:
# =========================================================
# 4. Load Preprocessed Modelling Inputs
# =========================================================

X_train_original = read_csv_safely(
    PREPROCESSING_DATASETS_PATH / "X_train_original.csv",
    "X_train_original"
)

X_test_original = read_csv_safely(
    PREPROCESSING_DATASETS_PATH / "X_test_original.csv",
    "X_test_original"
)

X_train_preprocessed = read_csv_safely(
    PREPROCESSING_DATASETS_PATH / "X_train_preprocessed.csv",
    "X_train_preprocessed"
)

X_test_preprocessed = read_csv_safely(
    PREPROCESSING_DATASETS_PATH / "X_test_preprocessed.csv",
    "X_test_preprocessed"
)

y_train = load_target_series(
    PREPROCESSING_DATASETS_PATH / "y_train.csv"
)

y_test = load_target_series(
    PREPROCESSING_DATASETS_PATH / "y_test.csv"
)

# =========================================================
# Remove Tracking Columns from Original Datasets
# =========================================================

TRACKING_COLUMNS = [
    "original_index"
]

X_train_model_original = (
    X_train_original
    .drop(columns=TRACKING_COLUMNS, errors="ignore")
    .copy()
)

X_test_model_original = (
    X_test_original
    .drop(columns=TRACKING_COLUMNS, errors="ignore")
    .copy()
)

# =========================================================
# Remove Tracking Columns from Preprocessed Matrices
# =========================================================

PREPROCESSED_TRACKING_COLUMNS = [
    "original_index",
    "index",
    "Unnamed: 0"
]

X_train_preprocessed = (
    X_train_preprocessed
    .drop(columns=PREPROCESSED_TRACKING_COLUMNS, errors="ignore")
    .copy()
)

X_test_preprocessed = (
    X_test_preprocessed
    .drop(columns=PREPROCESSED_TRACKING_COLUMNS, errors="ignore")
    .copy()
)

# =========================================================
# Input Summary
# =========================================================

print("X_train_original:", X_train_original.shape)
print("X_test_original:", X_test_original.shape)

print(
    "X_train_model_original:",
    X_train_model_original.shape
)

print(
    "X_test_model_original:",
    X_test_model_original.shape
)

print(
    "X_train_preprocessed:",
    X_train_preprocessed.shape
)

print(
    "X_test_preprocessed:",
    X_test_preprocessed.shape
)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

display(X_train_preprocessed.head())

X_train_original: (800, 10)
X_test_original: (200, 10)
X_train_model_original: (800, 9)
X_test_model_original: (200, 9)
X_train_preprocessed: (800, 29)
X_test_preprocessed: (200, 29)
y_train: (800,)
y_test: (200,)


,Age,Credit amount,Duration,Sex_female,Sex_male,Housing_free,Housing_own,Housing_rent,Saving accounts_Missing,Saving accounts_little,Saving accounts_moderate,Saving accounts_quite rich,Saving accounts_rich,Checking account_Missing,Checking account_little,Checking account_moderate,Checking account_rich,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others,Job_0,Job_1,Job_2,Job_3
0,1.000000,2.332558,1.500000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
1,0.357143,-0.586434,-0.500000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
2,-0.357143,1.183721,1.500000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
3,-0.285714,0.648450,1.500000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
4,-0.642857,2.391473,2.500000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000


In [35]:
# =========================================================
# 5. Validate Modelling Inputs
# =========================================================

validation_rows = [
    {"check": "X_train_preprocessed aligns with y_train", "passed": len(X_train_preprocessed) == len(y_train)},
    {"check": "X_test_preprocessed aligns with y_test", "passed": len(X_test_preprocessed) == len(y_test)},
    {"check": "Train/test preprocessed columns match", "passed": list(X_train_preprocessed.columns) == list(X_test_preprocessed.columns)},
    {"check": "Binary y_train target", "passed": sorted(pd.Series(y_train).unique().tolist()) == [0, 1]},
    {"check": "Binary y_test target", "passed": sorted(pd.Series(y_test).unique().tolist()) == [0, 1]},
    {"check": "No missing values in X_train_preprocessed", "passed": X_train_preprocessed.isna().sum().sum() == 0},
    {"check": "No missing values in X_test_preprocessed", "passed": X_test_preprocessed.isna().sum().sum() == 0},
]

input_validation_df = pd.DataFrame(validation_rows)

display(input_validation_df)

save_df(input_validation_df, "baseline_input_validation.csv")

if not input_validation_df["passed"].all():
    raise ValueError("One or more modelling input validation checks failed.")

,check,passed
0,X_train_preprocessed aligns with y_train,True
1,X_test_preprocessed aligns with y_test,True
2,Train/test preprocessed columns match,True
3,Binary y_train target,True
4,Binary y_test target,True
5,No missing values in X_train_preprocessed,True
6,No missing values in X_test_preprocessed,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/baseline_input_validation.csv


## 6. Load Fitted Preprocessor

The fitted preprocessor from `02` is used to build full pipeline models.  
If the preprocessor is not found, estimator-only models can still run, but full pipeline and DiCE readiness will fail.

In [36]:
# =========================================================
# 6. Load Fitted Preprocessor
# =========================================================

preprocessor_candidate_paths = [
    PREPROCESSING_MODELS_PATH / "preprocessor.pkl",
    PREPROCESSING_MODELS_PATH / "preprocessing_pipeline.joblib",
    PREPROCESSING_MODELS_PATH / f"{DATASET_NAME}_preprocessor.pkl",
]

fitted_preprocessor = None
preprocessor_path_found = None

for candidate_path in preprocessor_candidate_paths:
    if candidate_path.exists():
        fitted_preprocessor = joblib.load(candidate_path)
        preprocessor_path_found = candidate_path
        break

preprocessor_status_df = pd.DataFrame([{
    "preprocessor_found": fitted_preprocessor is not None,
    "preprocessor_path": str(preprocessor_path_found) if preprocessor_path_found is not None else None,
    "status": "Loaded" if fitted_preprocessor is not None else "Not found"
}])

display(preprocessor_status_df)

save_df(preprocessor_status_df, "preprocessor_load_status.csv")

,preprocessor_found,preprocessor_path,status
0,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/preprocessor_load_status.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/preprocessor_load_status.csv')

## 7. Define Candidate Models

All datasets use the same model families and the same core hyperparameters.  
This is critical for cross-domain comparability.

In [37]:
# =========================================================
# 7. Define Candidate Models
# =========================================================

models = {
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        random_state=RANDOM_STATE
    ),

    "decision_tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=RANDOM_STATE
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "mlp": MLPClassifier(
        hidden_layer_sizes=(50,),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        max_iter=1000,
        random_state=RANDOM_STATE
    )
}

if XGBOOST_AVAILABLE:
    models["xgboost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

pretty_model_names = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "mlp": "MLP"
}

model_registry_df = pd.DataFrame([
    {
        "model_id": model_id,
        "model_name": pretty_model_names.get(model_id, model_id),
        "model_object": type(model).__name__,
        "core_hyperparameters": str(model.get_params())
    }
    for model_id, model in models.items()
])

display(model_registry_df)

save_df(model_registry_df, "model_registry.csv")
save_df(model_registry_df, "model_registry.csv", folder=DATASETS_PATH)

,model_id,model_name,model_object,core_hyperparameters
0,logistic_regression,Logistic Regression,LogisticRegression,"{'C': 1.0, 'class_weight': None, 'dual': False..."
1,decision_tree,Decision Tree,DecisionTreeClassifier,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit..."
2,random_forest,Random Forest,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w..."
3,mlp,MLP,MLPClassifier,"{'activation': 'relu', 'alpha': 0.0001, 'batch..."
4,xgboost,XGBoost,XGBClassifier,"{'objective': 'binary:logistic', 'base_score':..."


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/model_registry.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/datasets/model_registry.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/datasets/model_registry.csv')

## 8. Train and Evaluate Estimator-Only Models

Estimator-only models are trained on the preprocessed feature matrix.  
These models are used later by SHAP and LIME.

In [38]:
# =========================================================
# 8. Train and Evaluate Estimator-Only Models
# =========================================================

performance_rows = []
prediction_rows = []
trained_models = {}

for model_id, base_model in models.items():

    model_name = pretty_model_names.get(model_id, model_id)

    print("=" * 100)
    print(f"Training estimator-only model: {model_name}")
    print("=" * 100)

    # -----------------------------------------------------
    # Create independent estimator instance
    # -----------------------------------------------------
    model = clone(base_model)

    start_time = time.time()

    model.fit(X_train_preprocessed, y_train)

    training_time_seconds = time.time() - start_time

    y_pred = model.predict(X_test_preprocessed)
    y_proba = get_model_probabilities(model, X_test_preprocessed)

    metrics = calculate_binary_metrics(
        y_test,
        y_pred,
        y_proba
    )

    performance_rows.append({
        "model_id": model_id,
        "model_name": model_name,
        "training_time_seconds": training_time_seconds,
        **metrics
    })

    for i in range(len(y_test)):
        prediction_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "test_instance_position": i,
            "true_class": int(
                y_test.iloc[i]
                if hasattr(y_test, "iloc")
                else y_test[i]
            ),
            "predicted_class": int(y_pred[i]),
            "predicted_probability": float(y_proba[i])
        })

    # -----------------------------------------------------
    # Store fitted estimator-only model
    # -----------------------------------------------------
    trained_models[model_id] = model

performance_df = pd.DataFrame(performance_rows)
prediction_summary_df = pd.DataFrame(prediction_rows)

display(performance_df)
display(prediction_summary_df.head())

Training estimator-only model: Logistic Regression
Training estimator-only model: Decision Tree
Training estimator-only model: Random Forest
Training estimator-only model: MLP
Training estimator-only model: XGBoost


,model_id,model_name,training_time_seconds,accuracy,precision,recall,f1_score,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate
0,logistic_regression,Logistic Regression,0.018544,0.735000,0.589744,0.383333,0.464646,0.761429,0.617207,124,16,37,23,0.114286,0.616667
1,decision_tree,Decision Tree,0.006912,0.680000,0.470588,0.533333,0.500000,0.739940,0.551867,104,36,28,32,0.257143,0.466667
2,random_forest,Random Forest,0.777399,0.745000,0.615385,0.400000,0.484848,0.769167,0.618334,125,15,36,24,0.107143,0.600000
3,mlp,MLP,2.513024,0.710000,0.517857,0.483333,0.500000,0.740238,0.552379,113,27,31,29,0.192857,0.516667
4,xgboost,XGBoost,0.122750,0.765000,0.632653,0.516667,0.568807,0.781429,0.656290,122,18,29,31,0.128571,0.483333


,model_id,model_name,test_instance_position,true_class,predicted_class,predicted_probability
0,logistic_regression,Logistic Regression,0,0,0,0.148239
1,logistic_regression,Logistic Regression,1,0,0,0.290699
2,logistic_regression,Logistic Regression,2,1,0,0.376362
3,logistic_regression,Logistic Regression,3,0,0,0.422767
4,logistic_regression,Logistic Regression,4,1,0,0.251970


## 9. Train and Evaluate Full Pipeline Models

Full pipeline models are trained on the original feature space and include:

```text
fitted preprocessor + classifier
```

These models are mainly used later by DiCE counterfactuals.

In [39]:
# =========================================================
# 9. Train and Evaluate Full Pipeline Models
# =========================================================

pipeline_performance_rows = []
pipeline_models = {}

if fitted_preprocessor is None:

    print(
        "Fitted preprocessor not found. "
        "Full pipeline models will be skipped."
    )

    pipeline_performance_df = pd.DataFrame()

else:

    for model_id, base_model in models.items():

        model_name = pretty_model_names.get(model_id, model_id)

        print("=" * 100)
        print(f"Training full pipeline model: {model_name}")
        print("=" * 100)

        # -------------------------------------------------
        # Create independent classifier instance
        # -------------------------------------------------
        pipeline_classifier = clone(base_model)

        pipeline_model = Pipeline(
            steps=[
                ("preprocessor", fitted_preprocessor),
                ("classifier", pipeline_classifier)
            ]
        )

        start_time = time.time()

        pipeline_model.fit(
            X_train_model_original,
            y_train
        )

        training_time_seconds = (
            time.time() - start_time
        )

        y_pred = pipeline_model.predict(
            X_test_model_original
        )

        y_proba = get_model_probabilities(
            pipeline_model,
            X_test_model_original
        )

        metrics = calculate_binary_metrics(
            y_test,
            y_pred,
            y_proba
        )

        pipeline_performance_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "training_time_seconds": training_time_seconds,
            **metrics
        })

        # -------------------------------------------------
        # Store fitted pipeline model
        # -------------------------------------------------
        pipeline_models[model_id] = pipeline_model

    pipeline_performance_df = pd.DataFrame(
        pipeline_performance_rows
    )

display(pipeline_performance_df)

Training full pipeline model: Logistic Regression
Training full pipeline model: Decision Tree
Training full pipeline model: Random Forest
Training full pipeline model: MLP
Training full pipeline model: XGBoost


,model_id,model_name,training_time_seconds,accuracy,precision,recall,f1_score,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate
0,logistic_regression,Logistic Regression,0.030582,0.735000,0.589744,0.383333,0.464646,0.761429,0.617207,124,16,37,23,0.114286,0.616667
1,decision_tree,Decision Tree,0.021657,0.680000,0.470588,0.533333,0.500000,0.739940,0.551867,104,36,28,32,0.257143,0.466667
2,random_forest,Random Forest,0.791164,0.745000,0.615385,0.400000,0.484848,0.769167,0.618334,125,15,36,24,0.107143,0.600000
3,mlp,MLP,2.083915,0.710000,0.517857,0.483333,0.500000,0.740238,0.552379,113,27,31,29,0.192857,0.516667
4,xgboost,XGBoost,0.093272,0.765000,0.632653,0.516667,0.568807,0.781429,0.656290,122,18,29,31,0.128571,0.483333


## 10. Business Priority Score

The business score uses dataset-specific weights from the configuration block.

In [40]:
# =========================================================
# 10. Business Priority Score
# =========================================================

BUSINESS_SCORE_WEIGHTS = CONFIG["business_score_weights"]

performance_df["business_priority_score"] = performance_df.apply(
    lambda row: business_priority_score(row, BUSINESS_SCORE_WEIGHTS),
    axis=1
)

performance_df["business_priority_score_pct"] = performance_df["business_priority_score"] * 100

performance_df["business_priority_rank"] = (
    performance_df["business_priority_score"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

performance_df = (
    performance_df
    .sort_values(["business_priority_rank", "model_id"])
    .reset_index(drop=True)
)

business_score_config_df = pd.DataFrame([
    {
        "metric": metric,
        "weight": weight,
        "weight_pct": weight * 100,
        "reason": CONFIG["business_score_reason"]
    }
    for metric, weight in BUSINESS_SCORE_WEIGHTS.items()
])

display(business_score_config_df)
display(performance_df)

save_df(business_score_config_df, "business_score_config.csv")

,metric,weight,weight_pct,reason
0,recall,0.500000,50.000000,German Credit prioritises recall because faili...
1,roc_auc,0.300000,30.000000,German Credit prioritises recall because faili...
2,precision,0.200000,20.000000,German Credit prioritises recall because faili...


,model_id,model_name,training_time_seconds,accuracy,precision,recall,f1_score,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate,business_priority_score,business_priority_score_pct,business_priority_rank
0,xgboost,XGBoost,0.122750,0.765000,0.632653,0.516667,0.568807,0.781429,0.656290,122,18,29,31,0.128571,0.483333,0.619293,61.929252,1
1,decision_tree,Decision Tree,0.006912,0.680000,0.470588,0.533333,0.500000,0.739940,0.551867,104,36,28,32,0.257143,0.466667,0.582766,58.276646,2
2,mlp,MLP,2.513024,0.710000,0.517857,0.483333,0.500000,0.740238,0.552379,113,27,31,29,0.192857,0.516667,0.567310,56.730952,3
3,random_forest,Random Forest,0.777399,0.745000,0.615385,0.400000,0.484848,0.769167,0.618334,125,15,36,24,0.107143,0.600000,0.553827,55.382692,4
4,logistic_regression,Logistic Regression,0.018544,0.735000,0.589744,0.383333,0.464646,0.761429,0.617207,124,16,37,23,0.114286,0.616667,0.538044,53.804396,5


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/business_score_config.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/business_score_config.csv')

## 11. Business Score Sensitivity Analysis

This section checks whether model ranking changes under alternative metric-weighting assumptions.

In [41]:
# =========================================================
# 11. Business Score Sensitivity Analysis
# =========================================================

sensitivity_scenarios = {
    "main_config_score": BUSINESS_SCORE_WEIGHTS,
    "equal_weight": {
        "recall": 1/3,
        "roc_auc": 1/3,
        "precision": 1/3
    },
    "recall_priority": {
        "recall": 0.50,
        "roc_auc": 0.30,
        "precision": 0.20
    },
    "precision_priority": {
        "recall": 0.20,
        "roc_auc": 0.30,
        "precision": 0.50
    },
    "ranking_priority": {
        "recall": 0.25,
        "roc_auc": 0.50,
        "precision": 0.25
    }
}

sensitivity_rows = []

for scenario_name, weights in sensitivity_scenarios.items():
    scenario_df = performance_df.copy()

    scenario_df["scenario_score"] = scenario_df.apply(
        lambda row: business_priority_score(row, weights),
        axis=1
    )

    scenario_df["scenario_rank"] = (
        scenario_df["scenario_score"]
        .rank(ascending=False, method="dense")
        .astype(int)
    )

    for _, row in scenario_df.iterrows():
        sensitivity_rows.append({
            "scenario": scenario_name,
            "model_id": row["model_id"],
            "model_name": row["model_name"],
            "scenario_score": row["scenario_score"],
            "scenario_score_pct": row["scenario_score"] * 100,
            "scenario_rank": row["scenario_rank"],
            "recall_weight": weights["recall"],
            "roc_auc_weight": weights["roc_auc"],
            "precision_weight": weights["precision"]
        })

business_score_sensitivity_df = pd.DataFrame(sensitivity_rows)

display(business_score_sensitivity_df.sort_values(["scenario", "scenario_rank", "model_id"]))

,scenario,model_id,model_name,scenario_score,scenario_score_pct,scenario_rank,recall_weight,roc_auc_weight,precision_weight
5,equal_weight,xgboost,XGBoost,0.643583,64.358277,1,0.333333,0.333333,0.333333
8,equal_weight,random_forest,Random Forest,0.594850,59.485043,2,0.333333,0.333333,0.333333
6,equal_weight,decision_tree,Decision Tree,0.581287,58.128735,3,0.333333,0.333333,0.333333
7,equal_weight,mlp,MLP,0.580476,58.047619,4,0.333333,0.333333,0.333333
9,equal_weight,logistic_regression,Logistic Regression,0.578168,57.816850,5,0.333333,0.333333,0.333333
0,main_config_score,xgboost,XGBoost,0.619293,61.929252,1,0.500000,0.300000,0.200000
1,main_config_score,decision_tree,Decision Tree,0.582766,58.276646,2,0.500000,0.300000,0.200000
2,main_config_score,mlp,MLP,0.567310,56.730952,3,0.500000,0.300000,0.200000
3,main_config_score,random_forest,Random Forest,0.553827,55.382692,4,0.500000,0.300000,0.200000
4,main_config_score,logistic_regression,Logistic Regression,0.538044,53.804396,5,0.500000,0.300000,0.200000


## 12. Ten-Fold Cross-Validation

Cross-validation is applied consistently across datasets using stratified folds.

In [42]:
# =========================================================
# 12. Ten-Fold Cross-Validation
# =========================================================

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1_score": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision"
}

cv_rows = []

for model_id, model in models.items():
    model_name = pretty_model_names.get(model_id, model_id)

    print(f"Running 10-fold CV for: {model_name}")

    cv_results = cross_validate(
        model,
        X_train_preprocessed,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    row = {
        "model_id": model_id,
        "model_name": model_name
    }

    for metric in scoring.keys():
        row[f"cv_mean_{metric}"] = cv_results[f"test_{metric}"].mean()
        row[f"cv_std_{metric}"] = cv_results[f"test_{metric}"].std()

    cv_rows.append(row)

cross_validation_summary_df = pd.DataFrame(cv_rows)

display(cross_validation_summary_df)

Running 10-fold CV for: Logistic Regression
Running 10-fold CV for: Decision Tree
Running 10-fold CV for: Random Forest
Running 10-fold CV for: MLP
Running 10-fold CV for: XGBoost


,model_id,model_name,cv_mean_accuracy,cv_std_accuracy,cv_mean_precision,cv_std_precision,cv_mean_recall,cv_std_recall,cv_mean_f1_score,cv_std_f1_score,cv_mean_roc_auc,cv_std_roc_auc,cv_mean_average_precision,cv_std_average_precision
0,logistic_regression,Logistic Regression,0.726250,0.036422,0.563791,0.092721,0.354167,0.093634,0.432910,0.096039,0.743824,0.058132,0.561079,0.077720
1,decision_tree,Decision Tree,0.698750,0.041627,0.489813,0.079845,0.433333,0.126656,0.456789,0.102841,0.714993,0.050023,0.476632,0.070002
2,random_forest,Random Forest,0.732500,0.038810,0.580483,0.100151,0.387500,0.087500,0.462497,0.088820,0.734933,0.062233,0.555094,0.087109
3,mlp,MLP,0.712500,0.049371,0.525765,0.106648,0.458333,0.089365,0.488105,0.092754,0.698661,0.040900,0.539567,0.071643
4,xgboost,XGBoost,0.746250,0.038750,0.602409,0.091835,0.470833,0.069846,0.526558,0.070581,0.763318,0.063148,0.617600,0.104471


## 13. Select Champion Model

The champion model is selected using the configured business-priority score.

In [43]:
# =========================================================
# 13. Select Champion Model
# =========================================================

champion_model_summary_df = (
    performance_df
    .sort_values(["business_priority_rank", "model_id"])
    .head()
    .copy()
)

display(champion_model_summary_df)

champion_model_id = champion_model_summary_df["model_id"].iloc[0]
champion_model_name = champion_model_summary_df["model_name"].iloc[0]

print(f"Champion model: {champion_model_name} ({champion_model_id})")

,model_id,model_name,training_time_seconds,accuracy,precision,recall,f1_score,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate,business_priority_score,business_priority_score_pct,business_priority_rank
0,xgboost,XGBoost,0.122750,0.765000,0.632653,0.516667,0.568807,0.781429,0.656290,122,18,29,31,0.128571,0.483333,0.619293,61.929252,1
1,decision_tree,Decision Tree,0.006912,0.680000,0.470588,0.533333,0.500000,0.739940,0.551867,104,36,28,32,0.257143,0.466667,0.582766,58.276646,2
2,mlp,MLP,2.513024,0.710000,0.517857,0.483333,0.500000,0.740238,0.552379,113,27,31,29,0.192857,0.516667,0.567310,56.730952,3
3,random_forest,Random Forest,0.777399,0.745000,0.615385,0.400000,0.484848,0.769167,0.618334,125,15,36,24,0.107143,0.600000,0.553827,55.382692,4
4,logistic_regression,Logistic Regression,0.018544,0.735000,0.589744,0.383333,0.464646,0.761429,0.617207,124,16,37,23,0.114286,0.616667,0.538044,53.804396,5


Champion model: XGBoost (xgboost)


## 14. Save Model Artefacts

Three artefact types are saved where available:

- estimator-only model;
- preprocessor copy;
- full pipeline model.

In [44]:
# =========================================================
# 14. Save Model Artefacts
# =========================================================

artifact_rows = []

for model_id, model in trained_models.items():
    model_name = pretty_model_names.get(model_id, model_id)

    estimator_path = MODELS_PATH / f"{model_id}.pkl"
    joblib.dump(model, estimator_path)

    artifact_rows.append({
        "model_id": model_id,
        "model_name": model_name,
        "artifact_type": "estimator_only",
        "artifact_path": str(estimator_path),
        "saved": estimator_path.exists()
    })

    if fitted_preprocessor is not None:
        preprocessor_copy_path = MODELS_PATH / f"{model_id}_preprocessor.pkl"
        joblib.dump(fitted_preprocessor, preprocessor_copy_path)

        artifact_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "artifact_type": "preprocessor_copy",
            "artifact_path": str(preprocessor_copy_path),
            "saved": preprocessor_copy_path.exists()
        })

    if model_id in pipeline_models:
        pipeline_path = MODELS_PATH / f"{model_id}_pipeline.pkl"
        joblib.dump(pipeline_models[model_id], pipeline_path)

        artifact_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "artifact_type": "full_pipeline",
            "artifact_path": str(pipeline_path),
            "saved": pipeline_path.exists()
        })

model_artifact_registry_df = pd.DataFrame(artifact_rows)

display(model_artifact_registry_df)

save_df(model_artifact_registry_df, "model_artifact_registry.csv")

,model_id,model_name,artifact_type,artifact_path,saved
0,logistic_regression,Logistic Regression,estimator_only,/content/drive/MyDrive/Master_Research/07_Resu...,True
1,logistic_regression,Logistic Regression,preprocessor_copy,/content/drive/MyDrive/Master_Research/07_Resu...,True
2,logistic_regression,Logistic Regression,full_pipeline,/content/drive/MyDrive/Master_Research/07_Resu...,True
3,decision_tree,Decision Tree,estimator_only,/content/drive/MyDrive/Master_Research/07_Resu...,True
4,decision_tree,Decision Tree,preprocessor_copy,/content/drive/MyDrive/Master_Research/07_Resu...,True
5,decision_tree,Decision Tree,full_pipeline,/content/drive/MyDrive/Master_Research/07_Resu...,True
6,random_forest,Random Forest,estimator_only,/content/drive/MyDrive/Master_Research/07_Resu...,True
7,random_forest,Random Forest,preprocessor_copy,/content/drive/MyDrive/Master_Research/07_Resu...,True
8,random_forest,Random Forest,full_pipeline,/content/drive/MyDrive/Master_Research/07_Resu...,True
9,mlp,MLP,estimator_only,/content/drive/MyDrive/Master_Research/07_Resu...,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/model_artifact_registry.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/model_artifact_registry.csv')

## 15. Compare Estimator-Only and Full Pipeline Performance

This confirms that estimator-only models and full pipeline models behave consistently.

In [45]:
# =========================================================
# 15. Compare Estimator-Only and Full Pipeline Performance
# =========================================================

comparison_cols = [
    "model_id",
    "model_name",
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc",
    "average_precision"
]

estimator_compare_df = performance_df[comparison_cols].copy()
estimator_compare_df["artifact_type"] = "estimator_only"

if not pipeline_performance_df.empty:
    pipeline_compare_df = pipeline_performance_df[comparison_cols].copy()
    pipeline_compare_df["artifact_type"] = "full_pipeline"

    artifact_performance_comparison_df = pd.concat(
        [estimator_compare_df, pipeline_compare_df],
        ignore_index=True
    )
else:
    artifact_performance_comparison_df = estimator_compare_df.copy()

display(artifact_performance_comparison_df.sort_values(["model_id", "artifact_type"]))

,model_id,model_name,accuracy,precision,recall,f1_score,roc_auc,average_precision,artifact_type
1,decision_tree,Decision Tree,0.680000,0.470588,0.533333,0.500000,0.739940,0.551867,estimator_only
6,decision_tree,Decision Tree,0.680000,0.470588,0.533333,0.500000,0.739940,0.551867,full_pipeline
4,logistic_regression,Logistic Regression,0.735000,0.589744,0.383333,0.464646,0.761429,0.617207,estimator_only
5,logistic_regression,Logistic Regression,0.735000,0.589744,0.383333,0.464646,0.761429,0.617207,full_pipeline
2,mlp,MLP,0.710000,0.517857,0.483333,0.500000,0.740238,0.552379,estimator_only
8,mlp,MLP,0.710000,0.517857,0.483333,0.500000,0.740238,0.552379,full_pipeline
3,random_forest,Random Forest,0.745000,0.615385,0.400000,0.484848,0.769167,0.618334,estimator_only
7,random_forest,Random Forest,0.745000,0.615385,0.400000,0.484848,0.769167,0.618334,full_pipeline
0,xgboost,XGBoost,0.765000,0.632653,0.516667,0.568807,0.781429,0.656290,estimator_only
9,xgboost,XGBoost,0.765000,0.632653,0.516667,0.568807,0.781429,0.656290,full_pipeline


## 16. Save Outputs

The saved outputs are used by SHAP, LIME, DiCE, EES and final results consolidation notebooks.

In [46]:
# =========================================================
# 16. Save Outputs
# =========================================================

save_df(performance_df, "baseline_model_performance.csv")
save_df(prediction_summary_df, "baseline_prediction_summary.csv")
save_df(pipeline_performance_df, "pipeline_model_performance.csv")
save_df(cross_validation_summary_df, "cross_validation_summary.csv")
save_df(business_score_sensitivity_df, "business_score_sensitivity_analysis.csv")
save_df(champion_model_summary_df, "champion_model_summary.csv")
save_df(artifact_performance_comparison_df, "artifact_performance_comparison.csv")
save_df(model_artifact_registry_df, "model_artifact_registry.csv")

save_df(performance_df, "baseline_model_performance.csv", folder=DATASETS_PATH)
save_df(prediction_summary_df, "baseline_prediction_summary.csv", folder=DATASETS_PATH)
save_df(pipeline_performance_df, "pipeline_model_performance.csv", folder=DATASETS_PATH)
save_df(cross_validation_summary_df, "cross_validation_summary.csv", folder=DATASETS_PATH)
save_df(business_score_sensitivity_df, "business_score_sensitivity_analysis.csv", folder=DATASETS_PATH)
save_df(champion_model_summary_df, "champion_model_summary.csv", folder=DATASETS_PATH)
save_df(artifact_performance_comparison_df, "artifact_performance_comparison.csv", folder=DATASETS_PATH)
save_df(model_artifact_registry_df, "model_artifact_registry.csv", folder=DATASETS_PATH)

save_excel(
    {
        "model_registry": model_registry_df,
        "input_validation": input_validation_df,
        "preprocessor_status": preprocessor_status_df,
        "baseline_performance": performance_df,
        "prediction_summary": prediction_summary_df,
        "pipeline_performance": pipeline_performance_df,
        "cv_summary": cross_validation_summary_df,
        "business_score_config": business_score_config_df,
        "business_sensitivity": business_score_sensitivity_df,
        "champion_model": champion_model_summary_df,
        "artifact_comparison": artifact_performance_comparison_df,
        "artifact_registry": model_artifact_registry_df
    },
    f"baseline_modeling_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "positive_class_label": CONFIG["positive_class_label"],
    "negative_class_label": CONFIG["negative_class_label"],
    "business_score_weights": CONFIG["business_score_weights"],
    "immutable_features": CONFIG["immutable_features"],
    "models": list(models.keys()),
    "standardisation_note": (
        "This baseline modelling notebook uses the same model families, core hyperparameters, "
        "evaluation metrics and output structure across German Credit, Pima Diabetes and COMPAS."
    )
}

with open(LOGS_PATH / "baseline_modeling_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "baseline_modeling_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All baseline modelling outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/baseline_model_performance.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/baseline_prediction_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/pipeline_model_performance.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/cross_validation_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/business_score_sensitivity_analysis.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/champion_model_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables/artifact_performanc

## 17. Interpretation Note

This notebook provides the predictive foundation for the full EES framework.

The trained models and saved artefacts are used by:

- SHAP interpretation;
- LIME interpretation;
- SHAP-LIME disagreement;
- DiCE counterfactual recourse;
- Fidelity scoring;
- Stability scoring;
- Actionability scoring;
- Regulatory alignment;
- Composite EES scoring;
- Final results consolidation.

Because the same model families and hyperparameters are used across datasets, later differences in explainability and EES results can be interpreted as domain/data differences rather than modelling-framework differences.

In [47]:
# =========================================================
# 17. Final Completion Message
# =========================================================

print("=" * 100)
print(f"03 Baseline Modelling completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Models saved to:   {MODELS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("\nKey outputs:")
print("- baseline_model_performance.csv")
print("- baseline_prediction_summary.csv")
print("- pipeline_model_performance.csv")
print("- cross_validation_summary.csv")
print("- business_score_sensitivity_analysis.csv")
print("- champion_model_summary.csv")
print("- model_artifact_registry.csv")
print("- artifact_performance_comparison.csv")
print(f"- baseline_modeling_{DATASET_NAME}_outputs.xlsx")

03 Baseline Modelling completed successfully for: German Credit
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/datasets
Models saved to:   /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/models
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/logs

Key outputs:
- baseline_model_performance.csv
- baseline_prediction_summary.csv
- pipeline_model_performance.csv
- cross_validation_summary.csv
- business_score_sensitivity_analysis.csv
- champion_model_summary.csv
- model_artifact_registry.csv
- artifact_performance_comparison.csv
- baseline_modeling_german_credit_outputs.xlsx
